# S5.G1-3 · G3 五臂对比 Santander：配对比较

按冻结的预注册与评估口径 v1（`modules/m6_evaluation/evaluation_protocol.md`）在开发评估集上一次性计算：
两层重抽样（训练 seed × 客户簇，2,000 次）的配对 AUROC 差、95% 区间与 p 值；可检出差异取两法较大者；四类结论；次要对比 Holm 校正（按预注册的完整次要家族 9 项计，偏保守）。
确定性臂（L0、L1、B_T2、A、L4）的 6 个 seed 打分相同，seed 维度方差为 0（freeze A-3）。输出只含汇总指标。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_g1_santander_run.yaml"
run = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
PRE_PATH = REPO_ROOT / run["preregistration"]
pre = yaml.safe_load(PRE_PATH.read_text(encoding="utf-8"))
fz = pre["freeze"]
ad = fz["arm_details"]


def sha(path):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:HASH_PREFIX_LEN]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("preregistration:", PRE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(PRE_PATH), "| status:", pre["status"])
print("seed:", run["seed"], "| seeds:", run["seeds"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
print("step:", run["step_id"], "|", run["purpose"])
mismatch = [p for p, h in fz["code_sha256"].items() if hashlib.sha256((REPO_ROOT / p).read_bytes()).hexdigest() != h]
print("冻结代码指纹核对：", "一致" if not mismatch else "不一致 " + str(mismatch))
assert pre["status"] == "frozen" and not mismatch
PRIV = pathlib.Path(os.environ[run["private_root_env"]])
SEED0 = run["seeds"][0]
MB, NJ = ad["xgboost_max_bin"], ad["n_jobs"]

config: modules/m5_modeling/configs/s5_g1_santander_run.yaml sha256:09834fbb00ea
preregistration: modules/m6_evaluation/configs/s6_p3_g3_preregistration.yaml sha256:c0a4bbfc9709 | status: frozen
seed: 11 | seeds: [11, 22, 33, 44, 55, 66]
git: f80f74f（工作区含本步未提交改动）
step: S5.G1 | 按冻结的 S6.P3 在 Santander 上训练 L0、L1（k=5/10/20）、B_T1、B_T2、A（SGB）、集中式代理 L4 与阴性对照 NC，并做配对比较
冻结代码指纹核对： 一致


In [2]:
from modules.m6_evaluation.components import metrics as MT
from modules.m6_evaluation.components import paired_eval as PE

arr = np.load(PRIV / run["arrays_file"])
local = np.load(PRIV / run["local_scores_file"])
sgb = np.load(PRIV / run["sgb_scores_file"])
TUNING = json.loads((PRIV / run["tuning_file"]).read_text(encoding="utf-8"))
y = arr["dev_eval__y"]
clusters = arr["dev_eval__cust"]
S = len(run["seeds"])


def as_matrix(v):
    v = np.asarray(v)
    return np.tile(v, (S, 1)) if v.ndim == 1 else v


SC = {a: as_matrix(local[a]) for a in local.files}
SC["A"] = as_matrix(sgb["dev_eval"])
SC["B_best"] = SC[TUNING["B_best"]]
n_pos, base = int(y.sum()), float(y.mean())
rows = []
for a in ["L0", "L1_k5", "L1_k10", "L1_k20", "B_T1", "B_T2", "A", "L4", "NC"]:
    m = SC[a]
    rows.append({"arm": a, "auroc": float(np.mean([MT.auc(y, s) for s in m])),
                 "lift_at_k": float(np.mean([MT.lift_at_k(y, s, run["top_k_share"]) for s in m])),
                 "recall_at_k": float(np.mean([MT.recall_at_k(y, s, run["top_k_share"]) for s in m])),
                 "seed_sd": float(np.std([MT.auc(y, s) for s in m]))})
arms = pd.DataFrame(rows)
arms["precision_at_k"] = arms["lift_at_k"] * base
print("开发评估集：%d 位客户，%d 个正例（正例率 %.4f）" % (len(y), n_pos, base))
print(arms.round(run["round_auc"]).to_string(index=False))

开发评估集：20351 位客户，586 个正例（正例率 0.0288）
   arm  auroc  lift_at_k  recall_at_k  seed_sd  precision_at_k
    L0 0.8227     3.5155       0.3515   0.0000          0.1012
 L1_k5 0.8219     3.2595       0.3259   0.0000          0.0939
L1_k10 0.8244     3.3449       0.3345   0.0000          0.0963
L1_k20 0.8201     3.1913       0.3191   0.0000          0.0919
  B_T1 0.8636     4.9348       0.4935   0.0054          0.1421
  B_T2 0.8400     3.9251       0.3925   0.0000          0.1130
     A 0.8671     5.2392       0.5239   0.0000          0.1509
    L4 0.8681     5.1368       0.5137   0.0000          0.1479
    NC 0.8190     3.4018       0.3402   0.0017          0.0980


**读法**（开发评估集 20,351 位客户、586 个正例，正例率 2.88%；名单取前 10%）：

- 内地单边 L0 的 AUROC 0.823；**L1（香港 k-匿名分段均值）三档都与 L0 相当**（0.820–0.824）。
- **B_T1（香港随机树嵌入，k 粗化后一次性发给内地）0.864，接近双向联邦 A 的 0.867**；B_T2（香港自有标签的分数）0.840。
- A（隐语 SGB）0.867 与集中式代理 L4 0.868 几乎相同；阴性对照 NC（打乱香港特征行）0.819，回到 L0 的水平。
- 前 10% 名单的精确率：L0 10.1%、L1 9.6%、B_T1 14.2%、A 15.1%。这是公开代理数据上的机制结果，**不能换算成汇丰的命中率或收益**。

In [3]:
rng = np.random.default_rng(run["seed"])
alpha, power = run["alpha"], run["power"]
delta = pre["primary"]["delta_of_interest"]
planned = [("primary", pre["primary"]["contrast"])] + [("secondary", c) for c in pre["secondary_contrasts"]]
planned.append(("check", ["A", "L4"]))
contrasts = [(k, c) for k, c in planned if c[0] in SC and c[1] in SC]
skipped = [" − ".join(c) for k, c in planned if (k, c) not in contrasts]
print("本批不含的对比（臂另立步骤，freeze A-9）：", skipped)
m_family = sum(1 for k, _ in planned if k == "secondary")   # Holm 按预注册的完整次要家族计，偏保守
out = []
for kind, (a, b) in contrasts:
    res = PE.paired_bootstrap(y, SC[a], SC[b], clusters, run["n_bootstrap"], alpha, rng)
    gap = PE.detectable_gap(res, n_pos, base, alpha, power, PE.MDE_CONSERVATIVE)
    out.append({"kind": kind, "contrast": "%s − %s" % (a, b), "diff": res["diff"], "ci_low": res["ci_low"],
                "ci_high": res["ci_high"], "p_value": res["p_value"], "detectable_gap": gap,
                "verdict": PE.verdict(res, gap, delta), "old_rule_pass": res["old_rule_pass"],
                "estimate_correlation": res["estimate_correlation"]})
con = pd.DataFrame(out)
sec = con["kind"] == "secondary"
order = con.loc[sec, "p_value"].sort_values().index
m_sec = m_family
holm, running = {}, 0.0
for rank, idx in enumerate(order):
    running = max(running, min(1.0, (m_sec - rank) * con.loc[idx, "p_value"]))
    holm[idx] = running
con["p_holm"] = pd.Series(holm)
print(con.drop(columns=["estimate_correlation"]).round(run["round_auc"]).to_string(index=False))
res_dir = REPO_ROOT / run["results_dir"]
res_dir.mkdir(parents=True, exist_ok=True)
arms.to_csv(res_dir / "arms_dev_eval.csv", index=False)
con.to_csv(res_dir / "contrasts_dev_eval.csv", index=False)
cost = json.loads((PRIV / "g3_santander_sgb_timing.json").read_text(encoding="utf-8"))
pd.DataFrame([{"n_customers": len(y), "n_positives": n_pos, "b_best": TUNING["B_best"],
               "sgb_fit_seconds": cost["fit_seconds"]}]).to_csv(res_dir / "run_summary.csv", index=False)
print("已写出公开汇总：", sorted(p.name for p in res_dir.iterdir()))

本批不含的对比（臂另立步骤，freeze A-9）： ['A_secure − A']


     kind        contrast    diff  ci_low  ci_high  p_value  detectable_gap       verdict  old_rule_pass  p_holm
  primary      A − L1_k10  0.0427  0.0325   0.0533    0.000          0.0237        可分辨的提升           True     NaN
secondary          A − L0  0.0444  0.0334   0.0555    0.000          0.0244        可分辨的提升           True   0.000
secondary     L1_k10 − L0  0.0017 -0.0034   0.0067    0.514          0.0123 在预声明尺度上无可分辨差异          False   1.000
secondary   B_T1 − L1_k10  0.0392  0.0281   0.0507    0.000          0.0250        可分辨的提升           True   0.000
secondary   B_T2 − L1_k10  0.0156  0.0066   0.0237    0.003          0.0206        可分辨的提升          False   0.021
secondary      A − B_best  0.0035 -0.0014   0.0092    0.168          0.0117 在预声明尺度上无可分辨差异          False   0.672
secondary         NC − L0 -0.0037 -0.0087   0.0006    0.097          0.0111 在预声明尺度上无可分辨差异          False   0.485
secondary  L1_k5 − L1_k10 -0.0025 -0.0067   0.0016    0.214          0.0099 在预声明尺度上无可分辨差异       

**读法**：

- **主对比 A − L1_k10 = +0.043，95% 区间 [0.033, 0.053]**，可分辨的提升，超过预注册的关心尺度 0.02。正例只有 586 个，可检出差异约 0.024，略大于 0.02；但区间远离 0，正例不足不影响这一结论。
- **L1 − L0 = +0.002，区间含 0**：L1 只拿到 A 相对 L0 增益的约 4%，与合成实验的 2%–16% 一致。k = 5 / 20 与 k = 10 没有可分辨差异。
- **A − B_best（B_T1）= +0.004，区间 [−0.001, 0.009] 含 0**：单向形态 B_T1 拿到双向增益的约 92%，两者在 0.02 尺度上无可分辨差异。
- B_T2 − L1 = +0.016，可分辨但小于 0.02。
- **阴性对照 NC − L0 = −0.004，区间含 0**：打乱香港特征后增益消失，A 的增益来自信息互补，不是容量或调参。
- **无损性核对 A − L4 = −0.001，区间含 0**：冻结修改 A-4、A-5 的代理前提成立，未触发新增的推翻条件。
- 旧判据一栏只作对照；确定性臂的 seed 维度没有方差，旧判据在这里没有信息。

## 结论（按预注册的决策映射）

1. **L3 相对 L1 有增量**（A − L1 = +0.043）→ S0.9 在形态 A 与 B_best 之间比较，进入 G4 盈亏平衡。
2. **单向形态可行**：B_T1 − L1 = +0.039，可分辨的提升；单向形态代价 A − B_T1 = +0.004，区间含 0。在这个构造的业务切分下，香港一次性发出的 k 粗化叶子编码，几乎拿到了双向联邦的全部增益。
3. **L1 在这里没有用**：香港按内地分段给出的聚合均值，几乎不带信息。
4. 阴性对照与无损性核对都通过。

**边界与提醒**：
- 公开竞赛数据上人为构造的两方切分（香港持产品持有与活跃度）；结论回答机制与相对大小，不代表汇丰真实效果（对外红线）。
- 开发评估集正例 586 个，低于预注册目标 700（停止规则 5，未追加样本）；这同时触发 DR-GOV-016 · D-2 的推翻条件，下一步先做 HSBC mock 三档（S5.G3）。
- B_T1 每棵树的叶子满足 k = 10，但每位客户 10 个叶子编码的**组合**可能接近唯一，k-匿名不能按组合成立；B_T1 是否构成个人信息、能否按单向形态处理，须另做披露评估与法务确认，不能直接当作「匿名统计」。
- 封存确认集未打开；本结论是开发证据。
- 技术侧自评，未经法务复核（W-001）。